# 11a — Distributed data and associative aggregation — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/11.md) · [Course map](../PLAN.md) · [Project](../../projects/stage11/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Compute mergeable summaries
- Recognize shuffle and skew
- Choose analytical storage formats


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Processing more data than fits in memory

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

A stream processes records incrementally rather than building one giant list. A batch is a bounded portion. An aggregate stores only the state needed for a summary, such as a count and sum per account.

An associative combination can merge partial results regardless of grouping: sums can be added, and `(sum,count)` pairs can be merged component-wise. Floating-point rounding can still vary with order; integer cents avoid that issue for this ledger.

Distributed processing partitions data among workers. A shuffle moves records so matching keys reach the same aggregation/join destination. A broadcast copies a small lookup table to workers. Skew means some keys receive much more data or work than others.

### Worked example: follow the values

Partition one has values `[10,20]`, mean 15; partition two has `[100]`, mean 100. Averaging those means gives 57.5, which is wrong. Merge sum/count: `(30+100)/(2+1)=130/3`, about 43.33.

### Pause and explain

How much aggregation state is needed for one million rows with only fifty account IDs?

<details><summary>Check your reasoning after trying</summary>

For simple per-account sums, about fifty accumulators rather than a million stored rows; memory grows with distinct keys.

</details>

### Common mistakes to check

Collecting all distributed rows back to the driver defeats the memory design. Include key cardinality in memory estimates.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

Spark’s driver coordinates work; executors process partitions. Transformations build a lazy plan; actions trigger execution. A narrow transformation can operate locally; grouping and large joins often require a shuffle across machines. More partitions do not automatically improve performance because scheduling, network, and serialization have costs.

To average partitions correctly, merge sums and counts. Averaging partition means is wrong when partition sizes differ. Associative summaries allow distributed combination. Skew occurs when a few keys receive most records; one task can dominate runtime. Inspect partition sizes and physical plans before tuning.

CSV and JSON are broadly readable but weakly typed and expensive to scan. Parquet is columnar, typed, compressed, and suitable for analytical column selection. Avro is row-oriented with schema support and is often useful for record/event interchange. Choose a format based on access patterns, schema evolution, and interoperability. A laptop local-mode Spark lab teaches APIs, not cluster operations at scale.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
partitions = [[10, 20], [100]]
print("Wrong mean of means:", sum(sum(p)/len(p) for p in partitions)/len(partitions))
print("Correct:", sum(map(sum, partitions))/sum(map(len, partitions)))


## Exercise 1: Mergeable average

Return (sum,count) for each partition, combine, and return the global mean; reject zero total count.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def distributed_mean(partitions):
    raise NotImplementedError


In [ ]:
assert abs(distributed_mean([[10, 20], [100]])-130/3) < 1e-12
assert distributed_mean([[], [3]]) == 3
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

Carry the denominator through aggregation.

</details>


## Exercise 2: Grouped partial sums

Merge a list of key→sum dictionaries into a new dictionary.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def merge_groups(partials):
    raise NotImplementedError


In [ ]:
assert merge_groups([{"a": 2}, {"a": 3, "b": 4}]) == {"a": 5, "b": 4}
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

Addition is associative for exact integers.

</details>


## Exercise 3: Skew ratio

Return largest nonnegative partition size divided by mean size; reject empty or all-zero input.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def skew_ratio(sizes):
    raise NotImplementedError


In [ ]:
assert skew_ratio([10, 10, 10]) == 1
assert skew_ratio([0, 0, 30]) == 3
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Compare maximum to mean, then inspect the full distribution.

</details>


## Independent transfer

Generate a million synthetic transactions in batches, compute grouped totals without loading everything into memory, and compare output with Spark. Explain shuffle, broadcast joins, and skew.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [11a interface and acceptance cases](../assignments/11a.md).
2. Copy the [blank starter](../assignments/starters/11a.py) to `work/assignments/11a.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 11a --solution work/assignments/11a.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/11a.md#11a-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 11a --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. Why is collect dangerous?

2. Why use an explicit schema?


## Reading and review

- [Primary reference 1](https://spark.apache.org/docs/latest/api/python/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
